# Day 067 — Exercise 2: Describe Image

**What you'll build:** `describe_image(img_b64, prompt, describe_fn=None)` — the core vision LLM call, with mock injection for testing.

**Why it matters:** This is the foundation all other exercises build on. The `describe_fn` injection pattern makes every downstream function testable without a running Ollama server — the same technique used with `process_fn` in Section 4.

In [ ]:
import io
import base64
from PIL import Image

def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

_test_img = Image.new('RGB', (100, 100), color=(255, 0, 0))
_test_b64 = image_to_base64(_test_img)


## Task

Implement `describe_image(img_b64, prompt, describe_fn=None) -> str`:

- If `describe_fn is not None`: call `describe_fn(img_b64, prompt)` and return
- Otherwise: call `ollama.chat(model='llava', messages=[{...}])` with the `images` key set to `[img_b64]`
- Return `resp['message']['content']`

All checks use a mock `describe_fn` — no Ollama server required.

## Your Implementation

In [ ]:
def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    """Send an image to a vision LLM and return its text response.

    Args:
        img_b64:     base64-encoded image string
        prompt:      question or instruction for the model
        describe_fn: optional callable(img_b64: str, prompt: str) -> str
                     If provided, calls this instead of Ollama (for testing).
                     If None, calls ollama.chat(model='llava', ...).
    Returns:
        Model response text
    """
    raise NotImplementedError


In [ ]:
def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    if describe_fn is not None:
        return describe_fn(img_b64, prompt)
    import ollama
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
    )
    return resp['message']['content']


## Automated checks

In [ ]:
score, total = 0, 5
try:
    captured = {}
    def _mock(b64, p):
        captured['b64']    = b64
        captured['prompt'] = p
        return f"Mock: saw image of length {len(b64)} with prompt '{p}'"

    result = describe_image(_test_b64, 'What colour is this?', describe_fn=_mock)
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    score += 1; print("\u2705 returns a string")

    assert len(result) > 0, "result should not be empty"
    score += 1; print("\u2705 non-empty result")

    assert captured.get('b64') == _test_b64, (
        "describe_fn should receive the img_b64 argument")
    score += 1; print("\u2705 mock receives the correct img_b64")

    assert captured.get('prompt') == 'What colour is this?', (
        f"Expected prompt 'What colour is this?', got {captured.get('prompt')!r}")
    score += 1; print("\u2705 mock receives the correct prompt")

    # Default prompt
    result2 = describe_image(_test_b64, describe_fn=lambda b, p: p)
    assert result2 == 'Describe this image.', (
        f"Default prompt mismatch: {result2!r}")
    score += 1; print("\u2705 default prompt is 'Describe this image.'")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    if describe_fn is not None:
        return describe_fn(img_b64, prompt)
    import ollama
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
    )
    return resp['message']['content']
```

**The mock-injection pattern:** checking `describe_fn is not None` first means any callable can override the Ollama path — a lambda, a class method, or a pre-recorded fixture. The production path (None → Ollama) is never reached in tests. This is identical to `process_fn` in Section 4 and will appear again in Days 69, 71, and 76.

</details>